# Stage D / NB 19 — external validation and domain shift (E9)

Protocol reference: family **E9**; referee points **R1.5** (no external validation) and
**R2.e** (generalisation unaddressed). Multiplicity family **F5**.

## What each sub-experiment is actually asking

| arm | cohort | the question |
| --- | --- | --- |
| E9a | X1 Montgomery normals | on radiographs with no disease, how often does it cry COVID? |
| E9b | X1 Montgomery TB | given *other* pathology, does it call the opacity COVID? |
| E9c | X2 cross-site PCR | does AUROC survive a different site, at the internal operating point and at a locally re-tuned one? |
| E9d | X3 RALO | does severity *rank* transfer under a different rubric? |
| E9e | X4 | only if the de-duplication audit is clean |
| E9f | MIDRC subgroups | does performance hold across sex, race, ethnicity? |
| E9g | X1–X4 | does a five-adapter ensemble beat a single fold's adapter? |

**E9b is the most clinically informative arm available and was absent before.** A system that
labels tuberculous opacity as COVID is not a severity scorer with a detection head; it is an
opacity detector wearing a label. That distinction decides how the paper may describe it.

## Three rules this notebook will not bend

**mRALE MAE is not computed on X3** (E9d). RALO is a different rubric on a different scale.
Spearman ρ and quadratic-weighted kappa after a monotone rank mapping are meaningful; MAE
against a rubric the model was never trained on is a number with no interpretation, and the
protocol says so explicitly. The code refuses rather than relying on discipline.

**X1 has no COVID positives.** Sensitivity and AUROC are undefined there. What X1 supports is
specificity and the false-positive rate, and nothing else.

**Thresholds are the internal ones.** E9c's headline applies the operating point selected on
*internal* inner validation, because that is what deploying the system would mean. The locally
re-tuned point is reported beside it as an upper bound — clearly labelled, never as the
headline.

## Test-time re-normalisation

A label-free intensity re-normalisation to the internal cohort's statistics is a legitimate
test-time adjustment and the referee explicitly asked whether one was considered. It is
evaluated as an arm (`+ttn`), not applied silently.

## Outputs (under `stage_D/nb19_external/`)
`external_metrics.csv`, `subgroup_metrics.csv`, `domain_shift_report.html`,
`e9b_tb_confusion.csv`, `e9c_threshold_transfer.csv`, `e9g_adapter_ensemble.csv`,
`f5_external_comparisons.csv`, 
`paired_comparisons_final.csv`, `multiplicity_families_final.json`, `gate_nb19.json`.

## 1. Imports, locked artifacts, and the external manifests

In [ ]:
import gc
import hashlib
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
from collections import Counter, OrderedDict, defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

# Metric definitions are shared with Stage B/C. Every number in the manuscript must come from
# the same code that produced the arm tables, or the tables and the statistics disagree.
_SEARCH = [Path.cwd(), Path.cwd().parent, Path.cwd().parent / "stage_B",
           Path.cwd().parent.parent / "notebooks" / "stage_B"]
for _candidate in _SEARCH:
    if (_candidate / "cxr_metrics.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError(f"cxr_metrics.py not found. Searched: {_SEARCH}")
import cxr_metrics as cm

# Stage D's own statistics module: bootstrap indices drawn once, DeLong, McNemar, Holm, TOST,
# and the rule that a p-value cannot exist without its metadata.
for _candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent / "stage_D",
                   Path.cwd().parent.parent / "notebooks" / "stage_D"]:
    if (_candidate / "stage_d_stats.py").is_file():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break
else:
    raise FileNotFoundError("stage_d_stats.py not found; it must sit beside these notebooks.")
import stage_d_stats as sd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

FALLBACK_STAGE_A = Path("/data/liangz2/openi/midrc/tetci_resubmit/stage_A")
for _candidate in [FALLBACK_STAGE_A / "nb00_environment" / "stage_a_paths.json",
                   Path.cwd() / "stage_a_paths.json",
                   Path.cwd().parent / "stage_A" / "nb00_environment" / "stage_a_paths.json"]:
    if _candidate.is_file():
        stage_paths = json.loads(_candidate.read_text(encoding="utf-8"))
        print("Path contract:", _candidate)
        break
else:
    raise FileNotFoundError("stage_a_paths.json not found. Run Stage A NB 00 first.")

PROJECT_ROOT = Path(stage_paths["project_root"])
STAGE_ROOT = Path(stage_paths["stage_root"])
STAGE_A_DIR = Path(stage_paths["stage_a_dir"])
STAGE_B_DIR = STAGE_ROOT / "stage_B"
STAGE_C_DIR = STAGE_ROOT / "stage_C"
STAGE_D_DIR = STAGE_ROOT / "stage_D"
STAGE_D_DIR.mkdir(parents=True, exist_ok=True)
NB01_DIR = Path(stage_paths["nb_output_dirs"]["nb01_inventory"])
NB02_DIR = Path(stage_paths["nb_output_dirs"]["nb02_folds"])
NB03_DIR = Path(stage_paths["nb_output_dirs"]["nb03_external"])
NB04_DIR = Path(stage_paths["nb_output_dirs"]["nb04_localization"])
FOLD_DEF_DIR = NB02_DIR / "fold_definitions"
MODEL_REVISIONS = stage_paths.get("model_revisions", {})

N_FOLDS = 5
N_BOOTSTRAP = sd.BOOTSTRAP_REPLICATES
MAX_SESSION_HOURS = 35.0      # Biowulf limit is 36 h; guard section boundaries
SESSION_DEADLINE = sd.make_session_deadline(MAX_SESSION_HOURS)

print("Stage D output:", STAGE_D_DIR)
print(f"Bootstrap: {N_BOOTSTRAP} patient-level replicates, seed {sd.BOOTSTRAP_SEED}")
print(f"Soft stop: {MAX_SESSION_HOURS:.1f} h after setup; checks occur between sections")

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 19 before code cell 3")

def load_folds():
    path = FOLD_DEF_DIR / "midrc_folds_v2.csv"
    if not path.is_file():
        raise FileNotFoundError(
            f"{path} not found. Run Stage A NB 02 first. Do NOT fall back to the legacy "
            "multi_task_CV folds: they leak at study level.")
    frame = pd.read_csv(path)
    frame["image_key"] = "MIDRC::" + frame["filename"].astype(str)
    return frame


# Every internal out-of-fold prediction file Stage B and Stage C can produce.
# (label, family, directory, filename). `family` drives the multiplicity families of 8.6.
INTERNAL_SOURCES = [
    ("A5_cxformer",        "E0",  STAGE_B_DIR / "nb05_frozen_encoder",     "predictions_frozen.jsonl"),
    ("E0g_conventional",   "E0",  STAGE_B_DIR / "nb06_conventional",       "predictions_conventional.jsonl"),
    ("E0_zeroshot",        "E0",  STAGE_B_DIR / "nb07_zeroshot",           "predictions_zeroshot.jsonl"),
    ("A6_biomedclip",      "E0",  STAGE_B_DIR / "nb08_biomedclip_entity",  "predictions_entity_probe.jsonl"),
    ("A2_medgemma_lora",   "E0",  STAGE_B_DIR / "nb09_medgemma_lora",      "predictions_medgemma_lora.jsonl"),
    ("A3_qwen_lora",       "E0",  STAGE_B_DIR / "nb10_qwen_lora",          "predictions_qwen_lora.jsonl"),
    ("A4_nvreason",        "E0",  STAGE_B_DIR / "nb11_nvreason",           "predictions_nvreason.jsonl"),
    ("E4_anatomy",         "E4",  STAGE_B_DIR / "nb12_anatomy_aware",      "anatomy_aware_predictions.jsonl"),
    ("E7_fusion",          "E7",  STAGE_C_DIR / "nb14_fusion",             "fusion_predictions.jsonl"),
]
EXTERNAL_SOURCES = [
    ("A5_cxformer",      STAGE_B_DIR / "nb05_frozen_encoder",    "external_predictions.jsonl"),
    ("E0g_conventional", STAGE_B_DIR / "nb06_conventional",      "external_predictions.jsonl"),
    ("A6_biomedclip",    STAGE_B_DIR / "nb08_biomedclip_entity", "external_predictions.jsonl"),
    ("A2_medgemma_lora", STAGE_B_DIR / "nb09_medgemma_lora",     "external_predictions.jsonl"),
    ("A3_qwen_lora",     STAGE_B_DIR / "nb10_qwen_lora",         "external_predictions.jsonl"),
]


def discover_arms(log=print):
    """
    Build the arm table from whatever Stage B and Stage C actually produced.

    Absence is recorded, never inferred: a notebook that has not run is a different thing from
    an arm that produced nothing, and the two have different remedies.
    """
    rows, availability = [], []
    for label, family, directory, filename in INTERNAL_SOURCES:
        path = directory / filename
        if not path.is_file():
            availability.append({"source": label, "family": family, "path": str(path),
                                 "status": "MISSING", "n_rows": 0, "n_arms": 0,
                                 "reason": "prediction file not found; notebook not yet run"})
            continue
        found = cm.read_jsonl(path)
        arms = sorted({str(r.get("arm", label)) for r in found})
        availability.append({"source": label, "family": family, "path": str(path),
                             "status": "OK", "n_rows": len(found), "n_arms": len(arms),
                             "reason": ""})
        for row in found:
            row["_source"] = label
            row["_family"] = family
            row["arm"] = str(row.get("arm", label))
            rows.append(row)

    # Stage C reasoner arms live in one file per roster.
    reasoner_dir = STAGE_C_DIR / "nb15_reasoner"
    reasoner_files = sorted(reasoner_dir.glob("reasoner_predictions_*.jsonl"))
    if reasoner_files:
        for path in reasoner_files:
            arm = path.stem.replace("reasoner_predictions_", "")
            found = cm.read_jsonl(path)
            family = "E1" if arm.startswith("E1") else "E7"
            availability.append({"source": f"reasoner/{arm}", "family": family,
                                 "path": str(path), "status": "OK", "n_rows": len(found),
                                 "n_arms": 1, "reason": ""})
            for row in found:
                row["_source"] = "reasoner"
                row["_family"] = family
                row["arm"] = arm
                rows.append(row)
    else:
        availability.append({"source": "reasoner", "family": "E1",
                             "path": str(reasoner_dir / "reasoner_predictions_*.jsonl"),
                             "status": "MISSING", "n_rows": 0, "n_arms": 0,
                             "reason": "NB 15 has not produced any roster predictions"})
    for entry in availability:
        log(f"  [{entry['status']:<7}] {entry['source']:<24} rows={entry['n_rows']:<7} "
            f"arms={entry['n_arms']}")
    return rows, pd.DataFrame(availability)

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 19 before code cell 4")

NB19_DIR = STAGE_D_DIR / "nb19_external"
NB19_DIR.mkdir(parents=True, exist_ok=True)
NB17_DIR = STAGE_D_DIR / "nb17_statistics"        # READ ONLY
NB18_DIR = STAGE_D_DIR / "nb18_calibration"       # READ ONLY

for required, owner in [(NB17_DIR / "all_metrics_with_ci.csv", "NB 17"),
                        (NB17_DIR / "paired_comparisons.csv", "NB 17"),
                        (NB17_DIR / "run_config.json", "NB 17"),
                        (NB03_DIR / "external_cohort_table.csv", "Stage A NB 03")]:
    if not required.is_file():
        raise FileNotFoundError(
            f"{required} not found — {owner} has not produced it. If the directory exists but "
            f"is empty the notebook has not finished; if it does not exist at all, the path "
            "contract is wrong and stage_a_paths.json is the thing to check.")

locked_metrics = pd.read_csv(NB17_DIR / "all_metrics_with_ci.csv")
nb17_config = json.loads((NB17_DIR / "run_config.json").read_text(encoding="utf-8"))
REFERENCE_ARM = nb17_config["reference_arm"]
STACKING_ARM = nb17_config["stacking_arm"]

nb18_gate_path = NB18_DIR / "gate_nb18.json"
nb18_config_path = NB18_DIR / "run_config.json"
operating_path = NB18_DIR / "operating_points.csv"
nb18_pair_path = NB18_DIR / "operating_point_comparisons.csv"
missing_nb18 = [path for path in [nb18_gate_path, nb18_config_path, operating_path,
                                  nb18_pair_path] if not path.is_file()]
if missing_nb18:
    raise FileNotFoundError(
        f"NB 19 requires the completed NB 18 contract; missing {missing_nb18}. "
        "Re-run NB 18 through its passing gate before starting NB 19.")
nb18_gate = json.loads(nb18_gate_path.read_text(encoding="utf-8"))
if not nb18_gate.get("passed", False):
    raise RuntimeError("NB 18 gate is not passing; external threshold transfer is blocked.")
nb18_config = json.loads(nb18_config_path.read_text(encoding="utf-8"))

# NB 18 reads NB 17's shared patient bootstrap read-only. A passing gate beside a stale
# operating_points.csv is not sufficient: all three fingerprints must identify the same
# frozen bootstrap and the endpoint-specific arm contract must agree.
expected_bootstrap = str(nb17_config.get("bootstrap_fingerprint") or "").strip()
observed_bootstraps = {
    "NB 18 gate": str(nb18_gate.get("bootstrap_fingerprint") or "").strip(),
    "NB 18 run configuration": str(nb18_config.get("bootstrap_fingerprint") or "").strip(),
}
if not expected_bootstrap:
    raise RuntimeError("NB 17 run_config.json has no bootstrap_fingerprint. Re-run NB 17.")
fingerprint_mismatch = {label: value for label, value in observed_bootstraps.items()
                        if value != expected_bootstrap}
if fingerprint_mismatch:
    raise RuntimeError(
        f"NB 18 does not belong to the completed NB 17 bootstrap: expected "
        f"{expected_bootstrap}, observed {fingerprint_mismatch}. Re-run NB 18 from the top.")
if str(nb18_config.get("reference_arm")) != str(REFERENCE_ARM):
    raise RuntimeError("NB 18 and NB 17 disagree on the locked reference arm.")
if str(nb18_config.get("mrale_stacking_arm")) != str(STACKING_ARM):
    raise RuntimeError("NB 18 and NB 17 disagree on the locked mRALE stacking arm.")
COVID_STACKING_ARM = str(nb18_config.get("covid_stacking_arm") or "").strip()
if not COVID_STACKING_ARM:
    raise RuntimeError("NB 18 did not record its endpoint-specific COVID stacking arm.")

operating_points = pd.read_csv(operating_path)
required_operating_columns = {"arm", "rule", "fold", "selected_on",
                              "applied_to", "transfer_threshold", "tp", "fp",
                              "tn", "fn"}
missing_columns = sorted(required_operating_columns - set(operating_points.columns))
if missing_columns:
    raise RuntimeError(f"NB 18 operating_points.csv lacks required columns: {missing_columns}")
pooled_nb18 = operating_points[operating_points["fold"].astype(str) == "pooled"]
if pooled_nb18.empty:
    raise RuntimeError("NB 18 operating_points.csv has no pooled transfer rows.")
if pooled_nb18.duplicated(["arm", "rule"]).any():
    duplicates = pooled_nb18.loc[pooled_nb18.duplicated(["arm", "rule"], keep=False),
                                 ["arm", "rule"]].drop_duplicates().to_dict("records")
    raise RuntimeError(f"NB 18 has duplicate pooled operating points: {duplicates[:5]}")
if not pooled_nb18["selected_on"].eq("inner_validation").all() \
        or not pooled_nb18["applied_to"].eq("held_out_fold").all():
    raise RuntimeError("NB 18 pooled thresholds violate the inner-validation/outer-test contract.")
if pd.to_numeric(pooled_nb18["transfer_threshold"], errors="coerce").isna().any():
    raise RuntimeError("NB 18 pooled operating points contain a missing transfer threshold.")

nb18_paired = pd.read_csv(nb18_pair_path)
required_pair_columns = {"comparison", "arm_a", "arm_b", "endpoint", "family",
                         "test", "paired_unit", "p_raw"}
missing_pair_columns = sorted(required_pair_columns - set(nb18_paired.columns))
if missing_pair_columns:
    raise RuntimeError(
        f"NB 18 operating_point_comparisons.csv lacks columns: {missing_pair_columns}")
nb18_f4 = nb18_paired[nb18_paired["family"].astype(str) == "F4"]
if len(nb18_f4) != 2 or int(nb18_config.get("mcnemar_rows", -1)) != 2:
    raise RuntimeError(
        f"NB 18 must contribute exactly two fixed-operating-point F4 tests; "
        f"found {len(nb18_f4)} CSV row(s) and run_config mcnemar_rows="
        f"{nb18_config.get('mcnemar_rows')!r}.")
print(f"NB 18 contract verified: {len(operating_points)} operating-point rows, "
      f"{len(nb18_f4)} fixed-OP F4 tests, bootstrap {expected_bootstrap}")

MANIFEST_DIR = NB03_DIR / "external_manifests"
COHORT_ROLE = {
    "X1": "Montgomery: normals (specificity) and TB (non-COVID pathology)",
    "X2": "cross-site PCR cohort",
    "X3": "RALO severity reference under a different rubric",
    "X4": "additional cohort, admitted only if the de-duplication audit is clean",
}
# Per protocol E9d: X3 supports RANK agreement only. Enforced, not remembered.
RANK_ONLY_COHORTS = {"X3"}
NO_POSITIVES_COHORTS = {"X1"}

manifests = {}
for cohort in ["X1", "X2", "X3", "X4"]:
    path = MANIFEST_DIR / f"{cohort}_manifest.csv"
    if path.is_file():
        manifests[cohort] = pd.read_csv(path)
        print(f"  {cohort}: {len(manifests[cohort]):,} images  ({COHORT_ROLE[cohort]})")
    else:
        print(f"  {cohort}: not prepared ({COHORT_ROLE[cohort]})")
if not manifests:
    raise RuntimeError("No external manifests found. Run Stage A NB 03 first.")

## 2. The membership guard is a precondition, not a footnote

An external cohort that overlaps training data is not external. NB 03 produced a graded-evidence
audit; this notebook refuses to score any cohort it flagged, and refuses to score X4 at all
unless its audit came back clean (E9e).

The graded tiers matter: NB 03 established that a bare perceptual-hash match at Hamming ≤ 3 is
**not** evidence on chest radiographs — the empirical collision rate is billions of times the
uniform null. HARD evidence (filename, SHA-256, DICOM UID) blocks; SOFT evidence warns.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 19 before code cell 6")

guard_path = NB03_DIR / "membership_guard.csv"
blocked_cohorts, guard = set(), pd.DataFrame()
if guard_path.is_file():
    guard = pd.read_csv(guard_path)
    print(guard.to_string(index=False))
    for row in guard.to_dict("records"):
        tier = str(row.get("evidence_tier", "")).upper()
        hits = int(row.get("n_hits", 0) or 0)
        clean = row.get("clean")
        if hits and tier == "HARD":
            blocked_cohorts.add(str(row["cohort"]))
        elif hits and tier == "SOFT":
            print(f"  {row['cohort']}: {hits} SOFT hit(s) — corroboration required before "
                  "these are treated as membership; NB 03's null test showed perceptual-hash "
                  "matches alone are not evidence on chest radiographs.")
        if clean is False and tier == "HARD":
            blocked_cohorts.add(str(row["cohort"]))
else:
    raise FileNotFoundError(
        f"{guard_path} is required before any external cohort can be admitted. Run NB 03; "
        "a missing leakage audit is not a clean leakage audit.")

if "X4" in manifests:
    audit = NB03_DIR / "x4_deduplication_audit.csv"
    x4_clean = False
    if audit.is_file():
        frame = pd.read_csv(audit)
        if "n_hits" not in frame.columns:
            raise RuntimeError("X4 de-duplication audit exists but has no n_hits column.")
        x4_clean = bool(len(frame) == 0 or pd.to_numeric(
            frame["n_hits"], errors="coerce").fillna(1).sum() == 0)
    if not x4_clean:
        blocked_cohorts.add("X4")
        print("\nX4 is EXCLUDED: protocol E9e admits it only if the de-duplication audit is "
              "clean, and it is not.")

if blocked_cohorts:
    print(f"\nBLOCKED COHORTS: {sorted(blocked_cohorts)} — overlap with training data, so they "
          "are not external and cannot support a generalisation claim.")
COHORTS = [c for c in manifests if c not in blocked_cohorts]
print(f"Cohorts admitted for external evaluation: {COHORTS}")

## 3. Collect external predictions

Stage B notebooks write `external_predictions.jsonl` with one row per (image, fold-adapter).
That shape is what makes **E9g** possible: a single fold's adapter is one row set, and the
five-adapter ensemble is their aggregate. External data is the only place the protocol permits
that ensemble, because the membership check came back empty.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 19 before code cell 8")

external_rows = []
for label, directory, filename in EXTERNAL_SOURCES:
    path = directory / filename
    if not path.is_file():
        print(f"  [MISSING] {label}: {path.name}")
        continue
    found = cm.read_jsonl(path)
    for row in found:
        row["_agent"] = label
        external_rows.append(row)
    print(f"  [OK]      {label}: {len(found):,} rows")

# The reasoner's external predictions, if Stage C produced any.
for path in sorted((STAGE_C_DIR / "nb15_reasoner").glob("external_predictions*.jsonl")):
    found = cm.read_jsonl(path)
    for row in found:
        row["_agent"] = "reasoner"
        external_rows.append(row)
    print(f"  [OK]      reasoner: {len(found):,} rows from {path.name}")

if not external_rows:
    print()
    print("No external predictions exist yet. Stage B's external inference has not been run, "
          "so E9 cannot be evaluated. This notebook will emit an empty result set and its "
          "gate will fail — which is the correct outcome, because R1.5 asked for external "
          "validation and there is none to report.")

INTERNAL_ARMS = set(locked_metrics["arm"].astype(str))


def normalize_external_arm(value):
    candidate = str(value)
    previous = None
    while candidate != previous:
        previous = candidate
        candidate = re.sub(r"(?:_external|_ensemble|_fold\d+)$", "", candidate)
    if candidate in INTERNAL_ARMS:
        return candidate
    matches = sorted(arm for arm in INTERNAL_ARMS
                     if candidate.startswith(arm) or arm.startswith(candidate))
    if len(matches) == 1:
        return matches[0]
    raise RuntimeError(f"External arm {value!r} does not map uniquely to a locked internal arm.")


manifest_keys = {}
for cohort, frame in manifests.items():
    if "image_key" in frame.columns:
        manifest_keys[cohort] = set(frame["image_key"].astype(str))

records = []
for row in external_rows:
    key = str(row.get("image_key", ""))
    cohort = str(row.get("cohort", key.split("::")[0] if "::" in key else "?"))
    if cohort not in COHORTS:
        continue
    if cohort in manifest_keys and key not in manifest_keys[cohort]:
        raise RuntimeError(f"Prediction {key} claims cohort {cohort} but is absent from its manifest.")
    arm = str(row.get("arm", row["_agent"]))
    fold = row.get("held_out_fold")
    records.append({
        "cohort": cohort, "subcohort": row.get("subcohort", cohort),
        "image_key": key, "agent": row["_agent"], "arm": arm,
        "adapter_fold": (int(fold) if fold is not None and not cm.is_missing(fold) else None),
        "ensemble_of_folds": row.get("ensemble_of_folds"),
        "base_arm": normalize_external_arm(arm),
        "patient": str(row.get("patient") or row.get("group_id")
                         or row.get("study_id") or key),
        "mrale_total": row.get("mrale_total"), "gt_mrale_total": row.get("gt_mrale_total"),
        "covid_pred": row.get("covid_pred"), "covid_score": row.get("covid_score"),
        "gt_covid": row.get("gt_covid"), "valid": bool(row.get("valid", False)),
    })

external = pd.DataFrame(records)
if len(external):
    identity = external[["base_arm", "image_key", "adapter_fold"]].copy()
    identity["adapter_fold"] = identity["adapter_fold"].fillna(-1)
    duplicate_external = identity.duplicated(keep=False)
    if duplicate_external.any():
        examples = identity.loc[duplicate_external].drop_duplicates().head(5).to_dict("records")
        raise RuntimeError(
            f"External prediction sources contain {int(duplicate_external.sum())} duplicate "
            f"arm/image/adapter rows, e.g. {examples}. Remove the stale duplicate source.")
print()
if len(external):
    print(f"External rows admitted: {len(external):,}")
    external_bootstrap = {}
    for cohort, group in external.groupby("cohort"):
        patients = sorted(group["patient"].astype(str).unique())
        external_bootstrap[cohort] = sd.PatientBootstrap(
            patients, n_replicates=N_BOOTSTRAP, seed=sd.BOOTSTRAP_SEED, log=lambda *a: None)
    print(external.groupby(["cohort", "subcohort"]).size().to_string())
    EXTERNAL_ARMS = sorted(external["base_arm"].unique())
    print(f"\nBase arms with external predictions: {EXTERNAL_ARMS}")
else:
    EXTERNAL_ARMS = []
    external_bootstrap = {}

## 4. E9a–E9d, one cohort at a time, each with only the metrics it can support

The dispatch below is the enforcement point. A cohort in `RANK_ONLY_COHORTS` never receives an
MAE, and a cohort in `NO_POSITIVES_COHORTS` never receives an AUROC or a sensitivity — the code
does not compute them, so they cannot leak into a table by accident.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 19 before code cell 10")

def cohort_metrics(rows, cohort, subcohort, arm, adapter):
    """Metrics this cohort can actually support, and no others."""
    entry = OrderedDict([("cohort", cohort), ("subcohort", subcohort), ("arm", arm),
                         ("adapter", adapter), ("n", len(rows)),
                         ("n_patients", len({str(r["patient"]) for r in rows}))])
    boot = external_bootstrap[cohort]
    positions = boot.patient_positions([r["patient"] for r in rows])
    truth_covid = [r["gt_covid"] for r in rows if r["gt_covid"] in {"Yes", "No"}]
    positives = sum(1 for t in truth_covid if t == "Yes")

    # ---- Detection ------------------------------------------------------------------------
    if truth_covid:
        labelled = [r for r in rows if r["gt_covid"] in {"Yes", "No"}]
        classification = cm.classification_metrics(
            [r["gt_covid"] for r in labelled], [r.get("covid_pred") for r in labelled],
            [r.get("covid_score") for r in labelled])
        entry["n_labelled"] = len(labelled)
        entry["n_positive"] = positives
        entry["specificity"] = classification.get("specificity")
        entry["false_positive_rate"] = (1 - classification["specificity"]
                                        if classification.get("specificity") is not None
                                        else None)
        for key in ["tp", "fp", "tn", "fn"]:
            entry[key] = classification.get(key)
        truth_binary = np.array([1 if r["gt_covid"] == "Yes" else 0
                                 for r in labelled], dtype=int)
        pred_binary = np.array([1 if r.get("covid_pred") == "Yes" else 0
                                for r in labelled], dtype=int)
        labelled_positions = boot.patient_positions([r["patient"] for r in labelled])
        negative = truth_binary == 0
        specificity_draws = boot.resample_statistic(
            labelled_positions, lambda w: sd.weighted_mean(
                (pred_binary[negative] == 0).astype(float), w[negative]))
        specificity_ci = sd.percentile_interval(specificity_draws)
        entry["specificity_ci_low"] = specificity_ci["ci_low"]
        entry["specificity_ci_high"] = specificity_ci["ci_high"]
        if cohort in NO_POSITIVES_COHORTS or positives == 0 or positives == len(labelled):
            # Undefined by construction; recording WHY keeps it out of the table honestly.
            entry["auroc"] = None
            entry["sensitivity"] = None
            entry["metrics_note"] = (f"{cohort} has {positives} positive(s) of "
                                     f"{len(labelled)}; AUROC and sensitivity are undefined")
        else:
            entry["auroc"] = classification.get("auroc")
            entry["auprc"] = classification.get("auprc")
            entry["sensitivity"] = classification.get("sensitivity")
            entry["balanced_accuracy"] = classification.get("balanced_accuracy")
            entry["brier"] = classification.get("brier")
            score_binary = np.array([cm.INVALID_COVID_SCORE if cm.is_missing(r.get("covid_score"))
                                     else float(r["covid_score"]) for r in labelled])
            auroc_draws = boot.resample_statistic(
                labelled_positions, lambda w: sd.weighted_auroc(truth_binary, score_binary, w))
            auroc_ci = sd.percentile_interval(auroc_draws)
            entry["auroc_ci_low"] = auroc_ci["ci_low"]
            entry["auroc_ci_high"] = auroc_ci["ci_high"]

    # ---- Severity --------------------------------------------------------------------------
    graded = [r for r in rows if not cm.is_missing(r.get("gt_mrale_total"))]
    if graded:
        truth = np.array([float(r["gt_mrale_total"]) for r in graded])
        predicted = np.array([np.nan if cm.is_missing(r.get("mrale_total"))
                              else float(r["mrale_total"]) for r in graded])
        usable = np.isfinite(predicted)
        entry["n_graded"] = len(graded)
        entry["severity_coverage"] = float(usable.mean())
        if cohort in RANK_ONLY_COHORTS:
            # Protocol E9d: rank agreement only. MAE against a different rubric is a number
            # with no interpretation, so it is not computed at all.
            entry["severity_metric_policy"] = "rank_only (different rubric; MAE not computed)"
            if usable.sum() >= 5 and np.std(truth[usable]) > 0 \
                    and np.std(predicted[usable]) > 0:
                try:
                    from scipy.stats import spearmanr
                    entry["spearman_rho"] = float(spearmanr(truth[usable],
                                                            predicted[usable])[0])
                except ImportError:
                    entry["spearman_rho"] = None
                # Prespecified monotone mapping: prediction percentile -> RALO extent 0..8.
                # It does not fit quantiles to this cohort's truth labels.
                ranks = pd.Series(predicted[usable]).rank(method="average").to_numpy() - 1.0
                mapped = 8.0 * ranks / max(len(ranks) - 1, 1)
                entry["qwk_after_rank_mapping"] = float(cm.quadratic_weighted_kappa(
                    np.rint(truth[usable]).astype(int), np.rint(mapped).astype(int),
                    min_rating=0, max_rating=8))
        else:
            entry["severity_metric_policy"] = "mae (same rubric)"
            severity = cm.mrale_metrics([
                {"gt_mrale_total": int(r["gt_mrale_total"]),
                 "mrale_total": (None if cm.is_missing(r.get("mrale_total"))
                                 else int(r["mrale_total"]))} for r in graded])
            for key in ["mae", "rmse", "qwk", "spearman_rho", "within1_accuracy", "coverage"]:
                if key in severity:
                    entry[f"severity_{key}"] = float(severity[key])
            graded_positions = boot.patient_positions([r["patient"] for r in graded])
            errors = np.where(usable, np.abs(np.nan_to_num(predicted, nan=0.0) - truth),
                              cm.INVALID_TOTAL_PENALTY)
            severity_draws = boot.resample_statistic(
                graded_positions, lambda w: sd.weighted_mean(errors, w))
            severity_ci = sd.percentile_interval(severity_draws)
            entry["severity_mae_ci_low"] = severity_ci["ci_low"]
            entry["severity_mae_ci_high"] = severity_ci["ci_high"]
    return entry


external_metric_rows = []
if len(external):
    for (cohort, subcohort, base_arm), group in external.groupby(
            ["cohort", "subcohort", "base_arm"]):
        # E9g: each fold's adapter separately, then the five-adapter ensemble.
        for adapter_fold, fold_group in group.groupby(group["adapter_fold"].fillna(-1)):
            if adapter_fold < 0:
                ensemble_sizes = pd.to_numeric(
                    fold_group["ensemble_of_folds"], errors="coerce").dropna().unique()
                if len(ensemble_sizes) > 1:
                    raise RuntimeError(
                        f"{base_arm}/{cohort}/{subcohort} has inconsistent fixed-ensemble "
                        f"sizes {sorted(ensemble_sizes.tolist())}.")
                label = (f"ensemble_of_{int(ensemble_sizes[0])}"
                         if len(ensemble_sizes) == 1 and int(ensemble_sizes[0]) > 1
                         else "single")
            else:
                label = f"fold{int(adapter_fold)}"
            external_metric_rows.append(
                cohort_metrics(fold_group.to_dict("records"), cohort, subcohort, base_arm,
                               label))

        folds_present = sorted({f for f in group["adapter_fold"].dropna().unique()})
        if folds_present == list(range(N_FOLDS)):
            ensemble = []
            for key, image_group in group.groupby("image_key"):
                rows = image_group.to_dict("records")
                totals = [float(r["mrale_total"]) for r in rows
                          if not cm.is_missing(r.get("mrale_total"))]
                scores = [float(r["covid_score"]) for r in rows
                          if not cm.is_missing(r.get("covid_score"))]
                ensemble.append({
                    "image_key": key,
                    "patient": rows[0]["patient"],
                    "mrale_total": (int(round(float(np.median(totals)))) if totals else None),
                    "covid_score": (float(np.mean(scores)) if scores else None),
                    "covid_pred": ("Yes" if scores and np.mean(scores) >= 0.5 else
                                   ("No" if scores else None)),
                    "gt_covid": rows[0]["gt_covid"],
                    "gt_mrale_total": rows[0]["gt_mrale_total"], "valid": bool(totals)})
            external_metric_rows.append(
                cohort_metrics(ensemble, cohort, subcohort, base_arm,
                               f"ensemble_of_{len(folds_present)}"))

external_metrics = pd.DataFrame(external_metric_rows)
if len(external_metrics):
    external_metrics.to_csv(NB19_DIR / "external_metrics.csv", index=False)
    columns = [c for c in ["cohort", "subcohort", "arm", "adapter", "n", "specificity",
                           "false_positive_rate", "auroc", "sensitivity", "spearman_rho",
                           "qwk_after_rank_mapping", "severity_mae"]
               if c in external_metrics.columns]
    print(external_metrics[columns].to_string(index=False))
else:
    print("No external metrics could be computed.")

## 5. E9b — does other pathology get called COVID?

The single most informative external arm, and the one the previous submission lacked entirely.

Montgomery's TB subcohort contains radiographs with real, visible, **non-COVID** opacity. If the
false-positive rate on TB is materially higher than on the normal subcohort, the detection head
is responding to opacity rather than to COVID — which is exactly what Stage B's internal
mechanism analysis suggested (max |ρ| between entity scores and PCR label was 0.082, against
0.775 for entity scores and mRALE).

That is a finding, not a defect to hide. It explains *why* detection fails and turns a null
result into a mechanism.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 19 before code cell 12")

e9b_rows = []
if len(external_metrics) and "X1" in COHORTS:
    x1 = external_metrics[external_metrics["cohort"] == "X1"]
    for arm, group in x1.groupby("arm"):
        normal = group[group["subcohort"].str.contains("normal", case=False, na=False)]
        tb = group[group["subcohort"].str.contains("tb", case=False, na=False)]
        if not len(normal) or not len(tb):
            continue
        shared_adapters = sorted(set(normal["adapter"]) & set(tb["adapter"]))
        if not shared_adapters:
            continue
        adapter = (next((value for value in shared_adapters if str(value).startswith("ensemble")), None)
                   or shared_adapters[0])
        normal_row = normal[normal["adapter"] == adapter].iloc[0]
        tb_row = tb[tb["adapter"] == adapter].iloc[0]
        fpr_normal = normal_row.get("false_positive_rate")
        fpr_tb = tb_row.get("false_positive_rate")
        if fpr_normal is None or fpr_tb is None:
            continue
        # Both subcohorts are PCR-negative and contain DIFFERENT images, so this is a
        # difference of two independent proportions -- not a paired test. McNemar would be
        # wrong here, and NB 17 deliberately does not adjust it.
        excess = float(fpr_tb) - float(fpr_normal)
        e9b_rows.append({
            "arm": arm, "adapter": normal_row.get("adapter"),
            "n_normal": int(normal_row["n"]), "n_tb": int(tb_row["n"]),
            "fpr_normal": round(float(fpr_normal), 4), "fpr_tb": round(float(fpr_tb), 4),
            "excess_fpr_on_tb": round(excess, 4),
            "fp_normal": normal_row.get("fp"), "fp_tb": tb_row.get("fp"),
            "reading": ("calls non-COVID opacity COVID: the detection head responds to opacity, "
                        "not to COVID" if excess > 0.10 else
                        "no material excess on TB" if abs(excess) <= 0.10 else
                        "fires LESS on TB opacity than on normals, which is hard to explain "
                        "and worth investigating before reporting")})

e9b = pd.DataFrame(e9b_rows)
if len(e9b):
    e9b.to_csv(NB19_DIR / "e9b_tb_confusion.csv", index=False)
    print(e9b[["arm", "n_normal", "n_tb", "fpr_normal", "fpr_tb", "excess_fpr_on_tb",
               "reading"]].to_string(index=False))
    worst = e9b.sort_values("excess_fpr_on_tb", ascending=False).iloc[0]
    print()
    print(f"Largest excess: {worst['arm']} fires on {worst['fpr_tb']:.1%} of TB radiographs "
          f"versus {worst['fpr_normal']:.1%} of normals ({worst['excess_fpr_on_tb']:+.1%}).")
    print("  These are two independent groups of PCR-negative images, so this is a difference")
    print("  of proportions, not a paired test. NB 17 does not adjust it; it is reported as a")
    print("  descriptive contrast with its counts.")
else:
    print("E9b could not be evaluated: X1 needs both a normal and a TB subcohort with "
          "predictions.")

## 6. E9c — the internal operating point, transferred

Two thresholds, and the order matters:

1. **The internal operating point**, selected on internal inner validation by NB 18. This is the
   headline, because it is what deploying the system would actually do.
2. **A locally re-tuned point**, selected on the external cohort itself. This is an **upper
   bound**, not a result — it uses the labels it is then scored against, and it is reported only
   to show how much of the gap is threshold drift rather than representation failure.

Presenting (2) as the external result is a common and serious overstatement. It is labelled in
the output column itself so it cannot be lifted into a table without its qualifier. When five
fold-specific adapters are available, the headline uses the **predeclared fold-0 adapter**; the
five-adapter ensemble is evaluated separately in E9g. If Stage B saved only a fixed,
precomputed ensemble, NB 19 preserves and declares that provenance instead of relabelling it
as a single model. NB 19 never selects an adapter using X2 labels or filesystem order.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 19 before code cell 14")

e9c_rows = []
e9c_issues = []
e9c_eligible_arms = []


def primary_external_adapter_rows(group, base_arm):
    """Return the predeclared single-adapter external rows used for E9c.

    With five CV adapters, fold 0 is fixed in advance. Choosing whichever adapter happens
    to occur first (or the best external adapter) would be order-dependent or label-driven.
    A precomputed ensemble is accepted when the upstream file explicitly records a single
    consistent `ensemble_of_folds` value. A genuinely non-CV arm is accepted only when
    every row has a null adapter-fold identifier and no ensemble declaration.
    """
    frame = group.copy()
    frame["adapter_fold_numeric"] = pd.to_numeric(frame["adapter_fold"], errors="coerce")
    folds_present = sorted(frame["adapter_fold_numeric"].dropna().astype(int).unique())
    ensemble_numeric = pd.to_numeric(frame["ensemble_of_folds"], errors="coerce")
    ensemble_sizes = sorted(ensemble_numeric.dropna().astype(int).unique())
    if folds_present and ensemble_sizes:
        raise RuntimeError(
            f"{base_arm}: rows declare both adapter folds and a precomputed ensemble.")
    if folds_present:
        if frame["adapter_fold_numeric"].isna().any():
            raise RuntimeError(
                f"{base_arm}: external predictions mix fold-specific and unlabelled adapters.")
        if 0 not in folds_present:
            raise RuntimeError(
                f"{base_arm}: fold-specific external predictions exist for {folds_present}, "
                "but predeclared fold 0 is absent; E9c will not select another fold post hoc.")
        selected = frame[frame["adapter_fold_numeric"] == 0].copy()
        source = "fold 0 (predeclared single adapter)"
    elif ensemble_sizes:
        if len(ensemble_sizes) != 1 or ensemble_sizes[0] <= 0:
            raise RuntimeError(
                f"{base_arm}: inconsistent ensemble_of_folds values {ensemble_sizes}.")
        if ensemble_numeric.isna().any():
            raise RuntimeError(
                f"{base_arm}: precomputed ensemble rows mix declared and missing sizes.")
        selected = frame.copy()
        source = (f"precomputed ensemble of {ensemble_sizes[0]} adapters (fixed upstream)"
                  if ensemble_sizes[0] > 1 else "single model (explicit upstream size 1)")
    else:
        selected = frame.copy()
        source = "single non-CV model"
    duplicates = selected.duplicated("image_key", keep=False)
    if duplicates.any():
        examples = selected.loc[duplicates, "image_key"].astype(str).unique()[:5].tolist()
        raise RuntimeError(
            f"{base_arm}: primary E9c adapter has duplicate image rows, e.g. {examples}.")
    return selected, source


if len(external) and "X2" in COHORTS and len(operating_points):
    pooled_points = pooled_nb18.copy()
    x2 = external[external["cohort"] == "X2"]
    for base_arm, group in x2.groupby("base_arm"):
        internal = pooled_points[(pooled_points["arm"] == base_arm)
                                 & (pooled_points["rule"] == "youden")]
        try:
            primary_group, adapter_source = primary_external_adapter_rows(group, base_arm)
        except RuntimeError as exc:
            e9c_issues.append(str(exc))
            continue
        rows = primary_group.to_dict("records")
        labelled = [r for r in rows if r["gt_covid"] in {"Yes", "No"}]
        scores = [r for r in labelled if not cm.is_missing(r.get("covid_score"))]
        if len(scores) < 30 or len({r["gt_covid"] for r in scores}) < 2:
            continue
        e9c_eligible_arms.append(base_arm)
        if len(internal) != 1:
            e9c_issues.append(
                f"{base_arm}: expected exactly one pooled NB 18 Youden transfer row, "
                f"found {len(internal)}.")
        truth = np.array([1.0 if r["gt_covid"] == "Yes" else 0.0 for r in scores])
        score = np.array([float(r["covid_score"]) for r in scores])

        auroc = sd.weighted_auroc(truth, score, np.ones(len(truth)))
        internal_auroc_rows = locked_metrics.loc[
            locked_metrics["arm"] == base_arm, "covid_auroc"]
        if len(internal_auroc_rows) > 1:
            e9c_issues.append(
                f"{base_arm}: NB 17 has {len(internal_auroc_rows)} internal AUROC rows.")
        internal_auroc = (float(internal_auroc_rows.iloc[0])
                          if len(internal_auroc_rows) == 1
                          and math.isfinite(float(internal_auroc_rows.iloc[0])) else None)

        def at(threshold):
            predicted = score >= threshold
            tp = int((predicted * truth).sum()); fp = int((predicted * (1 - truth)).sum())
            fn = int(((~predicted) * truth).sum()); tn = int(((~predicted) * (1 - truth)).sum())
            sensitivity = tp / (tp + fn) if tp + fn else float("nan")
            specificity = tn / (tn + fp) if tn + fp else float("nan")
            return {"tp": tp, "fp": fp, "tn": tn, "fn": fn, "sensitivity": sensitivity,
                    "specificity": specificity,
                    "balanced_accuracy": (sensitivity + specificity) / 2}

        if len(internal):
            threshold = float(internal.iloc[0]["transfer_threshold"])
            e9c_rows.append({"arm": base_arm, "cohort": "X2", "n": len(scores),
                             "adapter_source": adapter_source,
                             "threshold": round(threshold, 4),
                             "threshold_source": ("median of fold-specific internal "
                                                  "inner-validation thresholds (HEADLINE)"),
                             "auroc": round(auroc, 4),
                             "internal_auroc": internal_auroc,
                             "auroc_drop_vs_internal": (round(internal_auroc - auroc, 4)
                                                        if internal_auroc is not None else None),
                             **at(threshold)})
        # Locally re-tuned: an upper bound, computed on the labels it is scored against.
        best, best_value = 0.5, -np.inf
        for candidate in np.linspace(0, 1, 201):
            metrics = at(float(candidate))
            value = metrics["sensitivity"] + metrics["specificity"] - 1
            if math.isfinite(value) and value > best_value:
                best, best_value = float(candidate), value
        e9c_rows.append({"arm": base_arm, "cohort": "X2", "n": len(scores),
                         "adapter_source": adapter_source,
                         "threshold": round(best, 4),
                         "threshold_source": "re-tuned ON THIS COHORT (upper bound, not a "
                                             "result)",
                         "auroc": round(auroc, 4), "internal_auroc": internal_auroc,
                         **at(best)})

E9C_COLUMNS = ["arm", "cohort", "n", "adapter_source", "threshold",
               "threshold_source", "auroc", "internal_auroc",
               "auroc_drop_vs_internal", "tp", "fp", "tn", "fn",
               "sensitivity", "specificity", "balanced_accuracy"]
e9c = pd.DataFrame(e9c_rows).reindex(columns=E9C_COLUMNS)
# Always overwrite the artifact, including an empty but schema-valid result. This prevents
# a failed or partial rerun from leaving a stale successful threshold-transfer table behind.
e9c.to_csv(NB19_DIR / "e9c_threshold_transfer.csv", index=False)
if len(e9c):
    print(e9c[["arm", "adapter_source", "threshold_source", "threshold", "auroc",
               "internal_auroc", "sensitivity", "specificity",
               "balanced_accuracy"]].to_string(index=False))
    headline = e9c[e9c["threshold_source"].astype(str).str.contains("HEADLINE", na=False)]
    if len(headline) and headline["auroc_drop_vs_internal"].notna().any():
        drop = float(headline["auroc_drop_vs_internal"].astype(float).mean())
        print()
        print(f"Mean AUROC change from internal to X2: {-drop:+.4f}")
        print("  Read this beside the internal AUROC. If detection was already near chance")
        print("  internally, an unchanged external AUROC is not evidence of generalisation —")
        print("  it is evidence that the arm was uninformative in both places.")
else:
    print("E9c could not be evaluated: needs X2 predictions with scores and NB 18's operating "
          "points.")

## 7. E9f — subgroups

Per-subgroup MAE and balanced accuracy with patient-level intervals, and a flag on any subgroup
whose interval excludes the pooled estimate.

**What the metadata supports, and what it does not.** The fold definitions carry `sex`, `race`
and `ethnicity`. They do **not** carry age or an acquisition field, so the protocol's "age band"
and "portable-vs-fixed" subgroups cannot be computed. That absence is recorded rather than
quietly dropped: a referee asking why age is missing deserves "the field is not in the cohort"
rather than silence.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 19 before code cell 16")

folds = load_folds()
SUBGROUP_FIELDS = [f for f in ["sex", "race", "ethnicity", "quality_issue"]
                   if f in folds.columns]
UNAVAILABLE = [f for f in ["age_band", "acquisition_portable_vs_fixed"]
               if f not in folds.columns]
print(f"Subgroup fields available: {SUBGROUP_FIELDS}")
if UNAVAILABLE:
    print(f"NOT AVAILABLE in the cohort metadata: {UNAVAILABLE}")
    print("  The protocol asks for these; the fields do not exist in the MIDRC export used")
    print("  here. Report the absence rather than substituting a proxy.")

patient_of = dict(zip(folds["image_key"].astype(str), folds["group_id"].astype(str)))
def normalize_subgroup_value(value):
    """Return one deterministic, sortable label for mixed or missing metadata."""
    if value is None or (not isinstance(value, str) and pd.isna(value)):
        return "unknown"
    text = str(value).strip()
    return text if text and text.lower() not in {"nan", "none", "null"} else "unknown"


subgroup_of = {
    field: {str(key): normalize_subgroup_value(value)
            for key, value in zip(folds["image_key"], folds[field])}
    for field in SUBGROUP_FIELDS}
truth_mrale = dict(zip(folds["image_key"].astype(str),
                       folds["mrale_total_annotated"].astype(int)))
truth_covid = dict(zip(folds["image_key"].astype(str), folds["covid_positive"].astype(str)))

raw_rows, _ = discover_arms(log=lambda *a: None)
by_arm = defaultdict(list)
for row in raw_rows:
    key = str(row.get("image_key", ""))
    if key in truth_mrale:
        by_arm[row["arm"]].append(row)

subgroup_rows = []
if REFERENCE_ARM in by_arm:
    rows = by_arm[REFERENCE_ARM]
    all_patients = sorted({patient_of[str(r["image_key"])] for r in rows})
    boot = sd.PatientBootstrap(all_patients, n_replicates=N_BOOTSTRAP,
                               seed=sd.BOOTSTRAP_SEED, log=lambda *a: None)

    def subgroup_entry(subset, field, value):
        keys = [str(r["image_key"]) for r in subset]
        errors = np.array([
            cm.INVALID_TOTAL_PENALTY if cm.is_missing(r.get("mrale_total"))
            else abs(float(r["mrale_total"]) - truth_mrale[str(r["image_key"])])
            for r in subset])
        positions = boot.patient_positions([patient_of[k] for k in keys])
        draws = boot.resample_statistic(positions, lambda w: sd.weighted_mean(errors, w))
        interval = sd.percentile_interval(draws)
        entry = {"field": field, "value": value, "n_images": len(subset),
                 "n_patients": len({patient_of[k] for k in keys}),
                 "mrale_mae": round(float(errors.mean()), 4),
                 "mae_ci_low": round(interval["ci_low"], 4),
                 "mae_ci_high": round(interval["ci_high"], 4)}
        labelled = [r for r in subset if truth_covid[str(r["image_key"])] in {"Yes", "No"}]
        if len(labelled) >= 20:
            classification = cm.classification_metrics(
                [truth_covid[str(r["image_key"])] for r in labelled],
                [r.get("covid_pred") for r in labelled],
                [r.get("covid_score") for r in labelled])
            entry["balanced_accuracy"] = classification.get("balanced_accuracy")
            entry["auroc"] = classification.get("auroc")
            entry["n_positive"] = classification.get("tp", 0) + classification.get("fn", 0)
        return entry

    pooled = subgroup_entry(rows, "ALL", "pooled")
    subgroup_rows.append(pooled)
    for field in SUBGROUP_FIELDS:
        mapping = subgroup_of[field]
        groups = defaultdict(list)
        for r in rows:
            groups[mapping.get(str(r["image_key"]), "unknown")].append(r)
        # The explicit string sort is defensive against older cached fold tables that may
        # still contain a mixture of numeric and textual subgroup labels.
        for value, subset in sorted(groups.items(), key=lambda item: str(item[0])):
            if len(subset) < 25:
                continue
            entry = subgroup_entry(subset, field, value)
            # Flag a subgroup whose interval excludes the pooled point estimate.
            entry["excludes_pooled_mae"] = bool(
                entry["mae_ci_high"] < pooled["mrale_mae"]
                or entry["mae_ci_low"] > pooled["mrale_mae"])
            subgroup_rows.append(entry)

subgroups = pd.DataFrame(subgroup_rows)
if len(subgroups):
    subgroups.to_csv(NB19_DIR / "subgroup_metrics.csv", index=False)
    columns = [c for c in ["field", "value", "n_images", "n_patients", "mrale_mae",
                           "mae_ci_low", "mae_ci_high", "excludes_pooled_mae",
                           "balanced_accuracy"] if c in subgroups.columns]
    print()
    print(subgroups[columns].to_string(index=False))
    flagged = subgroups[subgroups.get("excludes_pooled_mae", False) == True]
    if len(flagged):
        print()
        print(f"{len(flagged)} subgroup(s) whose MAE interval EXCLUDES the pooled estimate:")
        for row in flagged.to_dict("records"):
            print(f"  {row['field']}={row['value']}: "
                  + sd.format_interval(row["mrale_mae"], row["mae_ci_low"],
                                       row["mae_ci_high"])
                  + f" vs pooled {sd.format_number(pooled['mrale_mae'])}")
        print("  These are descriptive, not multiplicity-controlled; with several subgroups")
        print("  some separation is expected by chance. Report them as observations that")
        print("  motivate a pre-registered subgroup analysis, not as established disparities.")
else:
    print("Subgroup analysis needs the reference arm's internal predictions.")

## 8. E9g — is a five-adapter ensemble worth it?

Permitted on external cohorts only, and only because the membership check came back empty. On
internal folds it would be leakage: fold *k*'s test images trained four of the five adapters.

The comparison is the ensemble against the best single adapter on the same images. An ensemble
that costs five model loads per image and buys nothing should be reported as such — protocol
§7.3 asks for the operational cost precisely so this trade-off is visible.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 19 before code cell 18")

e9g_rows = []
if len(external_metrics):
    for (cohort, subcohort, arm), group in external_metrics.groupby(
            ["cohort", "subcohort", "arm"]):
        ensemble = group[group["adapter"].astype(str).str.startswith("ensemble")]
        singles = group[group["adapter"].astype(str).str.startswith("fold")]
        if not len(ensemble) or not len(singles):
            continue
        for metric, better in [("severity_mae", "lower"), ("auroc", "higher"),
                               ("spearman_rho", "higher"), ("specificity", "higher")]:
            if metric not in group.columns:
                continue
            ensemble_value = ensemble.iloc[0].get(metric)
            single_values = singles[metric].dropna().astype(float)
            if cm.is_missing(ensemble_value) or not len(single_values):
                continue
            best_single = (float(single_values.min()) if better == "lower"
                           else float(single_values.max()))
            gain = (best_single - float(ensemble_value) if better == "lower"
                    else float(ensemble_value) - best_single)
            e9g_rows.append({
                "cohort": cohort, "subcohort": subcohort, "arm": arm, "metric": metric,
                "ensemble": round(float(ensemble_value), 4),
                "best_single_adapter": round(best_single, 4),
                "mean_single_adapter": round(float(single_values.mean()), 4),
                "spread_across_adapters": round(float(single_values.max()
                                                      - single_values.min()), 4),
                "ensemble_gain": round(gain, 4),
                "cost": f"{len(single_values)} model loads per image vs 1"})

e9g = pd.DataFrame(e9g_rows)
if len(e9g):
    e9g.to_csv(NB19_DIR / "e9g_adapter_ensemble.csv", index=False)
    print(e9g.to_string(index=False))
    positive = e9g[e9g["ensemble_gain"] > 0]
    print()
    print(f"The ensemble improves {len(positive)} of {len(e9g)} (cohort, metric) combinations.")
    print("  Read the gain against `spread_across_adapters`: a gain smaller than the spread")
    print("  between individual adapters is within the noise the ensemble is averaging over.")
    within_noise = e9g[e9g["ensemble_gain"].abs() < e9g["spread_across_adapters"]]
    if len(within_noise) >= len(e9g) / 2:
        print("  Most gains are inside that spread. On this evidence the ensemble buys little")
        print("  for five times the inference cost, and the paper should say so.")
else:
    print("E9g needs per-fold external predictions from all five adapters.")

# ---- F5 confirmatory tests and final cross-notebook multiplicity -------------------------
def bootstrap_two_sided(draws):
    finite = np.asarray(draws, dtype=float)
    finite = finite[np.isfinite(finite)]
    if not len(finite):
        return float("nan")
    lower = (int((finite <= 0).sum()) + 1) / (len(finite) + 1)
    upper = (int((finite >= 0).sum()) + 1) / (len(finite) + 1)
    return float(min(1.0, 2 * min(lower, upper)))


f5_rows = []
if len(external) and "X1" in COHORTS:
    x1 = external[external["cohort"] == "X1"]
    for arm, arm_group in x1.groupby("base_arm"):
        folds_here = sorted(pd.to_numeric(arm_group["adapter_fold"], errors="coerce").dropna().astype(int).unique())
        selected_fold = 0 if 0 in folds_here else (folds_here[0] if folds_here else None)
        chosen = arm_group[arm_group["adapter_fold"] == selected_fold] if selected_fold is not None else arm_group[arm_group["adapter_fold"].isna()]
        normal = chosen[chosen["subcohort"].astype(str).str.contains("normal", case=False)]
        tb = chosen[chosen["subcohort"].astype(str).str.contains("tb", case=False)]
        if not len(normal) or not len(tb):
            continue
        joined = pd.concat([normal.assign(group_name="normal"),
                            tb.assign(group_name="tb")], ignore_index=True)
        positions = external_bootstrap["X1"].patient_positions(joined["patient"])
        is_tb = joined["group_name"].eq("tb").to_numpy()
        fires = joined["covid_pred"].eq("Yes").to_numpy(dtype=float)
        def excess_fpr(weights):
            return (sd.weighted_mean(fires[is_tb], weights[is_tb])
                    - sd.weighted_mean(fires[~is_tb], weights[~is_tb]))
        draws = external_bootstrap["X1"].resample_statistic(positions, excess_fpr)
        interval = sd.percentile_interval(draws)
        effect = float(fires[is_tb].mean() - fires[~is_tb].mean())
        f5_rows.append({"comparison": f"{arm}: TB vs normal false-positive rate",
                        "arm_a": arm, "arm_b": arm, "endpoint": "X1 excess FPR on TB",
                        "family": "F5", "n_images": len(joined),
                        "n_patients": joined["patient"].nunique(), "delta": effect,
                        "ci_low": interval["ci_low"], "ci_high": interval["ci_high"],
                        "test": "patient-clustered bootstrap difference in proportions",
                        "paired_unit": "patient", "p_raw": bootstrap_two_sided(draws),
                        "adapter_fold": selected_fold})

# E9g: five-adapter ensemble versus the predeclared fold-0 adapter on identical images.
for (cohort, subcohort, arm), group in external.groupby(["cohort", "subcohort", "base_arm"]):
    if cohort in RANK_ONLY_COHORTS:
        continue
    folds_here = sorted(pd.to_numeric(group["adapter_fold"], errors="coerce").dropna().astype(int).unique())
    if folds_here != list(range(N_FOLDS)):
        continue
    paired_rows = []
    for key, image_group in group.groupby("image_key"):
        image_group = image_group.copy()
        image_group["adapter_fold_numeric"] = pd.to_numeric(
            image_group["adapter_fold"], errors="coerce")
        if image_group["adapter_fold_numeric"].duplicated().any():
            raise RuntimeError(f"Duplicate adapter prediction for {key} in {arm}.")
        per_fold = image_group.set_index("adapter_fold_numeric").reindex(range(N_FOLDS))
        truth_values = pd.to_numeric(per_fold["gt_mrale_total"], errors="coerce").dropna().unique()
        if len(truth_values) != 1 or pd.isna(per_fold.loc[0, "patient"]):
            continue
        truth_value = float(truth_values[0])
        totals = pd.to_numeric(per_fold["mrale_total"], errors="coerce")
        fold0_value = totals.loc[0]
        error_single = (abs(float(fold0_value) - truth_value)
                        if math.isfinite(float(fold0_value)) else cm.INVALID_TOTAL_PENALTY)
        error_ensemble = (abs(float(np.median(totals.to_numpy(dtype=float))) - truth_value)
                          if totals.notna().all() else cm.INVALID_TOTAL_PENALTY)
        paired_rows.append({"patient": str(per_fold.loc[0, "patient"]),
                            "difference": error_ensemble - error_single})
    if len(paired_rows) < 30:
        continue
    paired_frame = pd.DataFrame(paired_rows)
    positions = external_bootstrap[cohort].patient_positions(paired_frame["patient"])
    differences = paired_frame["difference"].to_numpy(dtype=float)
    draws = external_bootstrap[cohort].resample_statistic(
        positions, lambda w: sd.weighted_mean(differences, w))
    interval = sd.percentile_interval(draws)
    f5_rows.append({"comparison": f"{arm}: five-adapter ensemble vs fold 0",
                    "arm_a": f"{arm}:ensemble5", "arm_b": f"{arm}:fold0",
                    "endpoint": f"{cohort} mRALE MAE", "family": "F5",
                    "n_images": len(paired_frame),
                    "n_patients": paired_frame["patient"].nunique(),
                    "delta": float(differences.mean()), "ci_low": interval["ci_low"],
                    "ci_high": interval["ci_high"],
                    "test": "patient-clustered paired bootstrap",
                    "paired_unit": "patient", "p_raw": bootstrap_two_sided(draws)})

F5_COLUMNS = ["comparison", "arm_a", "arm_b", "endpoint", "family",
              "n_images", "n_patients", "delta", "ci_low", "ci_high",
              "test", "paired_unit", "p_raw", "adapter_fold"]
f5 = pd.DataFrame(f5_rows).reindex(columns=F5_COLUMNS)
f5.to_csv(NB19_DIR / "f5_external_comparisons.csv", index=False)
nb17_paired = pd.read_csv(NB17_DIR / "paired_comparisons.csv")
final_comparisons = pd.concat([nb17_paired, nb18_paired, f5], ignore_index=True, sort=False)
# NB 17 and NB 18 contain provisional multiplicity fields. Clear them before computing the
# only manuscript-reportable Holm adjustment across the now-complete F1--F5 families.
# Use explicit destination dtypes. `PValue.reportable` is a numeric p-value, not a Boolean;
# pandas >= 3 refuses the former implicit bool-to-float upcast (LossySetitemError).
final_comparisons["family_size"] = pd.Series(
    np.nan, index=final_comparisons.index, dtype="float64")
final_comparisons["adjusted"] = pd.Series(
    False, index=final_comparisons.index, dtype="bool")
final_comparisons["p_adjusted"] = pd.Series(
    np.nan, index=final_comparisons.index, dtype="float64")
final_comparisons["p_reportable"] = pd.Series(
    np.nan, index=final_comparisons.index, dtype="float64")
final_comparisons["manuscript_sentence"] = pd.Series(
    None, index=final_comparisons.index, dtype="object")
final_pvalues = []
final_pvalue_rows = []
for index, row in final_comparisons.iterrows():
    raw_p = pd.to_numeric(pd.Series([row.get("p_raw")]), errors="coerce").iloc[0]
    if not math.isfinite(float(raw_p)):
        continue
    item = sd.PValue(value=float(raw_p), test=str(row["test"]),
                     paired_unit=str(row["paired_unit"]), family=str(row["family"]),
                     family_size=1, effect=row.get("delta"),
                     effect_name=str(row.get("endpoint", "difference")),
                     ci_low=row.get("ci_low"), ci_high=row.get("ci_high"),
                     n=row.get("n_patients"),
                     note="Final Holm adjustment performed after NB 18 and NB 19.")
    final_pvalues.append(item)
    final_pvalue_rows.append((index, item))
sd.apply_holm_within_families(final_pvalues, alpha=0.05)
for index, item in final_pvalue_rows:
    final_comparisons.loc[index, "family_size"] = item.family_size
    final_comparisons.loc[index, "adjusted"] = item.adjusted
    final_comparisons.loc[index, "p_adjusted"] = item.adjusted_value
    final_comparisons.loc[index, "p_reportable"] = item.reportable
    final_comparisons.loc[index, "manuscript_sentence"] = item.sentence()
final_comparisons.to_csv(NB19_DIR / "paired_comparisons_final.csv", index=False)
family_counts = {family: int(sum(item.family == family for item in final_pvalues))
                 for family in sd.FAMILIES}
sd.write_json_atomic(NB19_DIR / "multiplicity_families_final.json",
                     {**sd.provenance_stamp("19_external_validation_and_domain_shift.ipynb"),
                      "family_counts": family_counts,
                      "source_files": ["NB17 paired_comparisons.csv",
                                         "NB18 operating_point_comparisons.csv",
                                         "NB19 f5_external_comparisons.csv"]})

# The decisive verdict must use the FINAL F4 family size, not NB 17's provisional one.
decisive_rows = final_comparisons[(final_comparisons["arm_a"] == STACKING_ARM)
                                  & (final_comparisons["arm_b"] == REFERENCE_ARM)
                                  & (final_comparisons["endpoint"] == "mRALE MAE")
                                  & (final_comparisons["reported_p_key"] == "p_bootstrap")]
if len(decisive_rows) != 1:
    raise RuntimeError(f"Expected one final decisive comparison, found {len(decisive_rows)}.")
decisive = decisive_rows.iloc[0]
delta = float(decisive["delta"])
adjusted_p = float(decisive["p_adjusted"])
significant = (math.isfinite(adjusted_p) and adjusted_p <= 0.05
               and (float(decisive["ci_low"]) > 0 or float(decisive["ci_high"]) < 0))
provisional_path = NB17_DIR / "decisive_comparison.json"
provisional = (json.loads(provisional_path.read_text(encoding="utf-8"))
               if provisional_path.is_file() else {})
if significant and delta > 0:
    verdict = "SUPPORTED: the reasoner has lower mRALE MAE than locked stacking."
elif significant:
    verdict = "REVERSED: locked stacking has lower mRALE MAE than the reasoner."
elif str(provisional.get("verdict", "")).startswith("EQUIVALENT"):
    verdict = ("EQUIVALENT FOR mRALE MAE by the predeclared TOST margin; "
               "no accuracy-superiority claim is supported.")
else:
    verdict = ("INCONCLUSIVE: final multiplicity-adjusted superiority is absent and "
               "equivalence was not established.")
sd.write_json_atomic(NB19_DIR / "decisive_comparison_final.json", {
    "comparison": decisive["comparison"], "endpoint": decisive["endpoint"],
    "delta": delta, "ci_low": decisive["ci_low"], "ci_high": decisive["ci_high"],
    "p_raw": decisive["p_raw"], "p_adjusted": adjusted_p,
    "family": decisive["family"], "family_size": int(decisive["family_size"]),
    "multiplicity_status": "FINAL after NB 18 and NB 19", "verdict": verdict})
print(f"Final multiplicity table: {len(final_comparisons)} tests; families {family_counts}")


## 9. Test-time intensity re-normalisation

The referee asked whether any test-time adjustment was considered. Re-normalising external
intensities to the internal cohort's statistics is label-free, so it is legitimate — it uses no
external labels and could be applied in deployment.

It cannot be evaluated from prediction files alone: it changes the input, so it requires
re-running inference with the adjustment applied. This cell reports whether Stage B produced
such an arm and, if not, says plainly that the arm is **declared but not executed** rather than
leaving a reader to assume it was tried.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 19 before code cell 20")

ttn_arms = sorted({a for a in EXTERNAL_ARMS if "ttn" in a.lower() or "renorm" in a.lower()})
ttn_status = {}
if ttn_arms:
    print(f"Test-time re-normalisation arms found: {ttn_arms}")
    for arm in ttn_arms:
        base = re.sub(r"[_+](ttn|renorm)\w*$", "", arm, flags=re.IGNORECASE)
        paired = external_metrics[external_metrics["arm"].isin([arm, base])]
        if len(paired):
            print(paired[[c for c in ["cohort", "arm", "adapter", "auroc", "specificity",
                                      "severity_mae"] if c in paired.columns]]
                  .to_string(index=False))
    ttn_status = {"executed": True, "arms": ttn_arms}
else:
    ttn_status = {
        "executed": False,
        "reason": ("No Stage B arm applied test-time intensity re-normalisation. The "
                   "adjustment changes the input, so it cannot be evaluated from stored "
                   "predictions; it needs an inference pass with the adjustment applied."),
        "manuscript_wording": ("Test-time intensity re-normalisation to the internal cohort "
                              "statistics is a legitimate label-free adjustment and was "
                              "specified in the protocol, but was not executed in this "
                              "revision. It is reported as declared-and-not-run rather than "
                              "as considered-and-rejected.")}
    print("No test-time re-normalisation arm was produced.")
    print(f"  {ttn_status['reason']}")
    print()
    print("  Manuscript wording:")
    print(f"  {ttn_status['manuscript_wording']}")

# X1's own domain shift is measurable from the manifests alone, and it is the point of E9a/E9b.
shift_rows = []
for cohort, frame in manifests.items():
    if cohort not in COHORTS:
        continue
    entry = {"cohort": cohort, "n_images": len(frame)}
    for column in ["width", "height"]:
        if column in frame.columns:
            values = pd.to_numeric(frame[column], errors="coerce").dropna()
            if len(values):
                entry[f"median_{column}"] = float(values.median())
    shift_rows.append(entry)
internal_dimensions = {}
if (NB01_DIR / "image_inventory.csv").is_file():
    inventory = pd.read_csv(NB01_DIR / "image_inventory.csv")
    for column in ["width", "height"]:
        if column in inventory.columns:
            internal_dimensions[f"median_{column}"] = float(
                pd.to_numeric(inventory[column], errors="coerce").dropna().median())
if shift_rows:
    shift = pd.DataFrame(shift_rows)
    if internal_dimensions:
        shift = pd.concat([pd.DataFrame([{"cohort": "MIDRC (internal)",
                                          "n_images": None, **internal_dimensions}]),
                           shift], ignore_index=True)
    shift.to_csv(NB19_DIR / "acquisition_shift.csv", index=False)
    print()
    print(shift.to_string(index=False))
    print("  Montgomery radiographs are markedly larger and differently processed than MIDRC's")
    print("  portable studies. That contrast IS the domain-shift signal in E9a/E9b, not a")
    print("  defect to normalise away silently.")

## 10. Domain shift report

A single HTML page a co-author can open without a notebook kernel: what was evaluated, on what,
with which caveats. It is generated from the tables above rather than written, so it cannot
drift from them.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 19 before code cell 22")

def html_table(frame, columns=None):
    if frame is None or not len(frame):
        return "<p><em>not available</em></p>"
    subset = frame[[c for c in (columns or frame.columns) if c in frame.columns]]
    return subset.to_html(index=False, float_format=lambda v: f"{v:.3f}", border=0,
                          classes="t", na_rep="--")


sections = [f"""<h2>Cohorts evaluated</h2>
<p>Admitted: <b>{', '.join(COHORTS) or 'none'}</b>.
Blocked by the membership guard: <b>{', '.join(sorted(blocked_cohorts)) or 'none'}</b>.</p>
{html_table(pd.read_csv(NB03_DIR / 'external_cohort_table.csv')
            if (NB03_DIR / 'external_cohort_table.csv').is_file() else None)}""",
            f"""<h2>E9a / E9b — specificity and non-COVID pathology</h2>
{html_table(e9b, ['arm', 'n_normal', 'n_tb', 'fpr_normal', 'fpr_tb', 'excess_fpr_on_tb',
                  'reading'])}
<p>A materially higher false-positive rate on tuberculous opacity than on normal radiographs
means the detection head is responding to opacity rather than to COVID.</p>""",
            f"""<h2>E9c — threshold transfer</h2>
{html_table(e9c, ['arm', 'threshold_source', 'threshold', 'auroc', 'internal_auroc',
                  'sensitivity', 'specificity'])}
<p>The re-tuned row uses the labels it is scored against and is an upper bound, not a
result.</p>""",
            f"""<h2>E9d — severity rank transfer</h2>
{html_table(external_metrics[external_metrics['cohort'].isin(RANK_ONLY_COHORTS)]
            if len(external_metrics) else None,
            ['cohort', 'arm', 'adapter', 'n_graded', 'spearman_rho',
             'qwk_after_rank_mapping', 'severity_metric_policy'])}
<p>MAE is deliberately absent: RALO is a different rubric on a different scale.</p>""",
            f"""<h2>E9f — subgroups</h2>
{html_table(subgroups, ['field', 'value', 'n_images', 'n_patients', 'mrale_mae',
                        'mae_ci_low', 'mae_ci_high', 'excludes_pooled_mae'])}
<p>Metadata unavailable for: {', '.join(UNAVAILABLE) or 'nothing'}.</p>""",
            f"""<h2>E9g — five-adapter ensemble</h2>
{html_table(e9g, ['cohort', 'arm', 'metric', 'ensemble', 'best_single_adapter',
                  'ensemble_gain', 'spread_across_adapters', 'cost'])}""",
            f"""<h2>Test-time re-normalisation</h2>
<p>{ttn_status.get('manuscript_wording', 'Executed as arm(s): '
                   + ', '.join(ttn_status.get('arms', [])))}</p>"""]

html = f"""<!doctype html><meta charset="utf-8">
<title>Domain shift and external validation</title>
<style>
body{{font:14px/1.55 -apple-system,Segoe UI,Roboto,sans-serif;max-width:1100px;margin:2rem auto;
padding:0 1rem;color:#1a1a1a}}
h1{{font-size:1.6rem}} h2{{font-size:1.1rem;margin-top:2rem;border-bottom:1px solid #ddd;
padding-bottom:.3rem}}
table.t{{border-collapse:collapse;font-size:12px;margin:.6rem 0}}
table.t th,table.t td{{border:1px solid #ddd;padding:3px 8px;text-align:right}}
table.t th{{background:#f5f5f5;text-align:left}} table.t td:first-child{{text-align:left}}
.note{{background:#fff8e1;border-left:3px solid #e0a800;padding:.6rem .9rem;margin:1rem 0}}
</style>
<h1>External validation and domain shift (E9)</h1>
<p>Generated {datetime.now(timezone.utc).isoformat()} from the tables in
<code>{NB19_DIR.name}/</code>. Reference arm: <b>{REFERENCE_ARM}</b>.</p>
<div class="note">Every number here comes from a CSV in this directory. Nothing is typed.
Confidence intervals use the patient-level bootstrap indices drawn once in NB 17.</div>
{''.join(sections)}
"""
(NB19_DIR / "domain_shift_report.html").write_text(html, encoding="utf-8")
print(f"domain_shift_report.html: {len(html):,} bytes")

## 11. Run configuration and gate

Blocking conditions include:

1. **No blocked cohort was scored.** A cohort that overlaps training data is not external, and
   a generalisation claim resting on one is worse than no claim.
2. **No MAE was computed on a rank-only cohort.** Protocol E9d, enforced by inspecting the
   emitted table rather than trusting the dispatch above.
3. **No AUROC or sensitivity on a single-class cohort.** Undefined quantities must not appear.
4. **External severity and AUROC rows carry patient-bootstrap intervals.**
5. **NB 18 provenance is locked.** Its gate, run configuration, threshold table, two F4
   McNemar rows, arm identities, and shared NB 17 bootstrap fingerprint must agree.
6. **E9c has exactly one headline for every eligible X2 arm.** Fold-specific rows use the
   predeclared fold-0 adapter; a fixed upstream ensemble remains explicitly labelled as an
   ensemble. No adapter is selected from external labels or file order.
7. **F5 is non-empty and every confirmatory p-value has its final Holm adjustment.**
8. **The central verdict is regenerated from the final F4 row.**

If no external predictions exist at all, the gate fails — R1.5 asked for external validation,
and an empty result set is the honest way to discover that it has not been produced yet.

In [ ]:
sd.check_session_deadline(SESSION_DEADLINE, "NB 19 before code cell 24")

failures, warnings = [], []

if not len(external_metrics):
    failures.append(
        "No external metrics were produced. Referee point R1.5 asked for external validation; "
        "Stage B's external inference (external_predictions.jsonl) has not been run for any "
        "arm, so family F5 is empty and no generalisation claim can be made.")

if len(external_metrics):
    scored_cohorts = set(external_metrics["cohort"])
    leaked = scored_cohorts & blocked_cohorts
    if leaked:
        failures.append(f"Cohort(s) {sorted(leaked)} were scored despite failing the "
                        "membership guard. They overlap training data and are not external.")
    else:
        print(f"Membership guard respected: scored {sorted(scored_cohorts)}, "
              f"blocked {sorted(blocked_cohorts) or 'none'}")

    rank_only = external_metrics[external_metrics["cohort"].isin(RANK_ONLY_COHORTS)]
    mae_columns = [c for c in rank_only.columns if c.startswith("severity_mae")]
    offending = [c for c in mae_columns if rank_only[c].notna().any()]
    if offending:
        failures.append(
            f"MAE was computed on rank-only cohort(s) {sorted(RANK_ONLY_COHORTS)} in columns "
            f"{offending}. Protocol E9d forbids it: RALO is a different rubric on a different "
            "scale, so the number has no interpretation.")
    else:
        print(f"Rank-only policy respected for {sorted(RANK_ONLY_COHORTS)}: Spearman and QWK "
              "after monotone rank mapping, no MAE.")

    single_class = external_metrics[external_metrics["cohort"].isin(NO_POSITIVES_COHORTS)]
    bad = single_class[single_class["auroc"].notna()] if "auroc" in single_class.columns \
        else pd.DataFrame()
    if len(bad):
        failures.append(f"{len(bad)} row(s) report an AUROC on a cohort with no positives. "
                        "The quantity is undefined and must not appear.")
    else:
        print(f"Single-class policy respected for {sorted(NO_POSITIVES_COHORTS)}: specificity "
              "and false-positive rate only.")

    same_rubric = external_metrics[~external_metrics["cohort"].isin(RANK_ONLY_COHORTS)]
    severity_rows = pd.DataFrame()
    if "severity_mae" in same_rubric:
        severity_rows = same_rubric[same_rubric["severity_mae"].notna()]
    if len(severity_rows):
        if "severity_mae_ci_low" not in severity_rows or severity_rows["severity_mae_ci_low"].isna().any():
            failures.append("External same-rubric severity rows are missing patient-level CIs.")
    auroc_rows = pd.DataFrame()
    if "auroc" in external_metrics:
        auroc_rows = external_metrics[external_metrics["auroc"].notna()]
    if len(auroc_rows):
        if "auroc_ci_low" not in auroc_rows or auroc_rows["auroc_ci_low"].isna().any():
            failures.append("External AUROC rows are missing patient-level bootstrap CIs.")

if not len(f5):
    failures.append("Multiplicity family F5 is empty; no confirmatory external comparison was run.")
elif family_counts.get("F5", 0) != len(f5):
    failures.append("F5 family size does not match the emitted external comparisons.")

if len(final_comparisons):
    confirmatory = final_comparisons[final_comparisons["family"] != "EXPLORATORY"]
    missing_adjustment = confirmatory[confirmatory["p_adjusted"].isna()]
    if len(missing_adjustment):
        failures.append(f"{len(missing_adjustment)} confirmatory p-values lack final Holm adjustment.")
if not (NB19_DIR / "decisive_comparison_final.json").is_file():
    failures.append("The decisive reasoner-vs-stacking verdict was not regenerated from "
                    "the final F4-adjusted p-value.")

# ---- Warnings ------------------------------------------------------------------------------
absent = [c for c in ["X1", "X2", "X3", "X4"] if c not in manifests]
if absent:
    warnings.append(f"Cohort(s) {absent} were never prepared by NB 03, so those E9 arms do not "
                    "exist. State which cohorts were available rather than implying all four.")

if len(e9b):
    worst = e9b.sort_values("excess_fpr_on_tb", ascending=False).iloc[0]
    if float(worst["excess_fpr_on_tb"]) > 0.10:
        warnings.append(
            f"E9b: {worst['arm']} fires on {float(worst['fpr_tb']):.1%} of TB radiographs vs "
            f"{float(worst['fpr_normal']):.1%} of normals. The detection head responds to "
            "non-specific opacity; this supports, but does not by itself prove, opacity-driven "
            "COVID decisions. Report the effect and its uncertainty with that qualifier.")
else:
    warnings.append("E9b was not evaluated. It is the most clinically informative external arm "
                    "available and its absence should be stated.")

if not ttn_status.get("executed"):
    warnings.append("Test-time re-normalisation was declared but not executed; use the wording "
                    "in run_config.json rather than implying it was tried and rejected.")

if len(subgroups):
    flagged = subgroups[subgroups.get("excludes_pooled_mae", False) == True]
    if len(flagged):
        warnings.append(
            f"{len(flagged)} subgroup(s) have an MAE interval excluding the pooled estimate: "
            + ", ".join(f"{r['field']}={r['value']}" for r in flagged.to_dict("records")[:4])
            + ". Descriptive only — not multiplicity-controlled, and some separation is "
              "expected across this many subgroups.")
if UNAVAILABLE:
    warnings.append(f"Protocol subgroups {UNAVAILABLE} cannot be computed: the fields are not "
                    "in the cohort metadata.")

headline = (e9c[e9c["threshold_source"].astype(str).str.contains("HEADLINE", na=False)]
            if len(e9c) else pd.DataFrame(columns=["arm"]))
if e9c_issues:
    failures.append("E9c contract violation(s): " + " | ".join(e9c_issues[:5]))
if "X2" in COHORTS:
    eligible = set(e9c_eligible_arms)
    observed = set(headline["arm"].astype(str)) if len(headline) else set()
    duplicate_headlines = (headline.groupby("arm").size() > 1) if len(headline) else pd.Series(dtype=bool)
    if not eligible:
        failures.append("E9c has no eligible X2 arm with at least 30 scored, two-class rows.")
    elif eligible != observed or duplicate_headlines.any():
        failures.append(
            f"E9c requires exactly one transferred-threshold headline per eligible X2 arm; "
            f"eligible={sorted(eligible)}, observed={sorted(observed)}, "
            f"duplicates={duplicate_headlines[duplicate_headlines].index.tolist()}.")

sd.write_json_atomic(NB19_DIR / "run_config.json", sd.provenance_stamp(
    "19_external_validation_and_domain_shift.ipynb",
    {"cohorts_admitted": COHORTS, "cohorts_blocked": sorted(blocked_cohorts),
     "cohorts_absent": absent, "reference_arm": REFERENCE_ARM,
     "mrale_stacking_arm": STACKING_ARM, "covid_stacking_arm": COVID_STACKING_ARM,
     "nb18_gate_path": str(nb18_gate_path),
     "nb18_run_config_path": str(nb18_config_path),
     "nb18_operating_points_path": str(operating_path),
     "nb18_operating_point_comparisons_path": str(nb18_pair_path),
     "nb18_bootstrap_fingerprint": expected_bootstrap,
     "e9c_primary_adapter_policy": ("fold 0 when fold-specific rows exist; accept an "
                                      "explicit fixed upstream ensemble; otherwise require "
                                      "a single non-CV model"),
     "e9c_eligible_arms": sorted(set(e9c_eligible_arms)),
     "e9c_contract_issues": e9c_issues,
     "external_arms": EXTERNAL_ARMS, "n_external_rows": int(len(external)),
     "rank_only_cohorts": sorted(RANK_ONLY_COHORTS),
     "no_positive_cohorts": sorted(NO_POSITIVES_COHORTS),
     "subgroup_fields": SUBGROUP_FIELDS, "subgroup_fields_unavailable": UNAVAILABLE,
     "test_time_renormalisation": ttn_status,
     "multiplicity_family": "F5", "final_family_counts": family_counts,
     "final_comparison_artifact": str(NB19_DIR / "paired_comparisons_final.csv"),
     "external_bootstrap_fingerprints": {cohort: boot.fingerprint
                                             for cohort, boot in external_bootstrap.items()},
     "family_note": ("NB 19 finalizes Holm adjustment after combining NB 17 tests, "
                     "NB 18 fixed-operating-point McNemar rows, and NB 19 F5 external tests.")}))


def report(title, messages):
    print(title)
    for message in messages or []:
        print("  -", message)
    if not messages:
        print("  none")


print()
report("WARNINGS", warnings)
print()
report("FAILURES", failures)
sd.write_json_atomic(NB19_DIR / "gate_nb19.json",
                     {"passed": not failures, "failures": failures,
                      "warnings": warnings, "nb18_gate": str(nb18_gate_path),
                      "nb18_bootstrap_fingerprint": expected_bootstrap})
if failures:
    detail = "\n".join(f"  [{i + 1}] {m}" for i, m in enumerate(failures))
    raise AssertionError(f"NB 19 gate failed with {len(failures)} blocking issue(s):\n{detail}")
print()
print("NB 19 gate: PASSED")